In [1]:
from mpramnist.DNASynBench import SimpleMotifDataset
from mpramnist.DNASynBench import LitModel_DNASyn

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

from mpramnist import transforms as t

import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

BATCH_SIZE = 128
NUM_WORKERS = 8

In [2]:
# Define set of transforms
train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.Seq2Tensor(),])

# Generating task 1 (simple classification)
dataset = SimpleMotifDataset()
dataset.generate(motif='AGTCGACT',
    length=200,
    n_seqs=10000,
    train_size=0.7,
    random_state=42)

train_dataset = dataset.get_split("train", transform=train_transform)
val_dataset = dataset.get_split("val", transform=val_test_transform)
test_dataset = dataset.get_split("test", transform=val_test_transform)

# Encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

100%|██████████████████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 20261.52it/s]


In [4]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_DNASyn(
    model=model, loss=nn.MSELoss(), weight_decay=1e-4, lr=1e-3, print_each=5
)
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=5,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)



Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/vastre34/miniconda3/envs/mpramnist/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To

Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...


-------------------------------------------------------------------------------
| Epoch: 4 | Val Loss: 0.00060 | Val Pearson: 0.99842 | Train Pearson: 0.98694 
-------------------------------------------------------------------------------



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing: |                                                                                        | 0/? [00:00…

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss          0.0005382741801440716
      test_pearson          0.9985734820365906
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.0005382741801440716, 'test_pearson': 0.9985734820365906}]